# Taller E2-02 — Inferencia sobre una media, una mediana y una proporción: control de calidad textil

**Estadística II (29006) — Ingeniería Industrial, UIS**

## Contexto

TelaFina S.A.S. es una empresa manufacturera que produce uniformes corporativos para varios clientes industriales. Desde hace dos años mantiene un contrato de suministro con una cadena de restaurantes que le exige cumplir tres parámetros de calidad en cada lote de producción, definidos conjuntamente durante la negociación del contrato. El primero tiene que ver con el gramaje de la tela, es decir, su peso por metro cuadrado: el cliente exige que el gramaje promedio se mantenga en 180 gramos por metro cuadrado, porque de ese valor depende la resistencia y la sensación al tacto que se espera de los uniformes. El segundo parámetro es el número de defectos de costura visibles por prenda; el contrato establece que, en una prenda representativa, no debería presentarse más de un defecto de costura, ya que un número mayor incrementa notablemente las devoluciones de los empleados que usan el uniforme a diario. El tercer parámetro tiene que ver con el color de la tela: cada prenda debe coincidir, dentro de una tolerancia acordada, con el tono Pantone de referencia que el cliente aprobó al inicio del contrato, y se exige que al menos el 95% de las prendas de cada lote cumplan esa tolerancia de color, porque los uniformes son visibles al público y cualquier variación notoria de tono afecta la imagen de marca del cliente.

Para verificar estos tres parámetros, el departamento de aseguramiento de calidad de TelaFina inspecciona periódicamente una muestra de las prendas terminadas de cada lote. El protocolo interno establece que, de la lista completa de prendas producidas, se debe seleccionar aleatoriamente un subconjunto para inspección detallada, usando un generador de números aleatorios sobre el identificador de cada prenda — en teoría, cada prenda del lote tiene la misma probabilidad de ser elegida, aproximando un muestreo aleatorio simple. En la práctica, la empresa reconoce que hay limitaciones logísticas (las prendas se apilan por orden de producción y no siempre es igual de sencillo acceder físicamente a cualquiera de ellas), pero el protocolo fue diseñado buscando que esa aleatorización sea lo más cercana posible a la ideal.

La gerencia de operaciones evalúa distintas acciones dependiendo de lo que arrojen estos análisis. Si el gramaje se desvía del estándar, la opción más costosa es detener la línea para recalibrar la máquina tejedora (varias horas de producción detenida); una alternativa más barata es ajustar la tensión del telar en el siguiente lote, sin detener el actual. Si el número de defectos de costura resulta mayor al esperado, la empresa contempla contratar un inspector de costura adicional de tiempo completo, o programar mantenimiento preventivo a las máquinas de coser del turno correspondiente. Y si la conformidad de color cae por debajo del umbral pactado, TelaFina se expone a una multa contractual, o a que el cliente rechace el lote completo y exija su reproceso — aunque la gerencia también podría negociar con el cliente una tolerancia de color ligeramente más amplia a cambio de un descuento en el precio del lote, si la desviación resulta ser pequeña.

Estas tres variables se miden sobre la misma prenda inspeccionada:

| Variable | Tipo | Descripción |
|---|---|---|
| `gramaje_tela` | Numérica continua | Peso de la tela por metro cuadrado (g/m²), medido con una balanza de precisión sobre una muestra estándar cortada de la prenda. |
| `defectos_costura` | Numérica discreta | Número de defectos visibles de costura (hilos sueltos, puntadas irregulares, costuras desalineadas) detectados en la inspección visual de la prenda. |
| `color_conforme` | Categórica binaria (Si/No) | Si el tono de color de la tela, medido con un colorímetro, está dentro de la tolerancia acordada frente al Pantone de referencia del cliente. |

## Instrucciones generales

- Trabajas con una muestra propia (MAS) de prendas — tu código estudiantil determina cuáles te tocaron.
- En las preguntas de prueba de hipótesis, guarda siempre el **resultado completo de la prueba** (el objeto que devuelve la función) además del estadístico y el p-valor por separado — la verificación automática revisa los tres.
- Antes de correr `ottr::check`, sube `anexos.zip` a tu sesión de Colab (trae la carpeta de tests y la base de datos).


In [ ]:
# Si subiste anexos.zip (y no las carpetas/archivos ya descomprimidos), descomenta y corre esta linea:
# unzip("anexos.zip")

if (!dir.exists("tests") || !file.exists("calidad_textil.csv")) {
  cat("AVISO: falta la carpeta 'tests' o 'calidad_textil.csv'. Sube anexos.zip y descomprimelo antes de continuar.\n")
}

# Instala ottr (el paquete que corre las auto-verificaciones) solo si hace falta
if (!requireNamespace("ottr", quietly = TRUE)) install.packages("ottr")

In [ ]:
# ============================================================
# CONFIGURACIÓN INICIAL — NO MODIFIQUES ESTA CELDA (salvo la línea indicada)
# ============================================================

# --> Reemplaza el valor de abajo por tu código estudiantil (solo números)
codigo_estudiante <- 0000000

reseed <- function(offset) {
  set.seed(codigo_estudiante + offset)
  RNGkind("Mersenne-Twister", "Inversion", "Rejection")
}

datos_completos <- read.csv("calidad_textil.csv", stringsAsFactors = FALSE)

reseed(1)
tamano_muestra <- 150
indices_muestra <- sample(seq_len(nrow(datos_completos)), size = tamano_muestra)
muestra <- datos_completos[indices_muestra, ]

cat("Tu muestra tiene", nrow(muestra), "prendas.\n")
head(muestra)


---
## Parte 1 — ¿Cumple la tela con el gramaje pactado en el contrato?

El área de calidad debe reportarle a gerencia si el gramaje de la tela sigue cumpliendo el valor pactado con el cliente: 180 g/m². Antes de aplicar una prueba de medias, conviene verificar si es razonable asumir que el gramaje se distribuye normalmente en la población de prendas: esa es la validación que hace la prueba de Shapiro-Wilk.

En una prueba de Shapiro-Wilk:
- **H0:** los datos provienen de una distribución normal.
- **H1:** los datos no provienen de una distribución normal.

**Función recomendada:** `shapiro.test()` — [documentación](https://www.rdocumentation.org/packages/stats/versions/3.6.2/topics/shapiro.test).


**Pregunta 1.1 (código):** antes de decidir qué prueba usar para comparar el gramaje promedio contra el estándar del contrato, verifica si es razonable asumir que `gramaje_tela` se distribuye normalmente en tu muestra. Aplica la prueba de Shapiro-Wilk y guarda: el resultado completo en `prueba_shapiro_gramaje`, el estadístico de prueba en `estadistico_shapiro_gramaje`, y el p-valor en `p_valor_shapiro_gramaje`.

In [ ]:
# BEGIN SOLUTION
prueba_shapiro_gramaje <- shapiro.test(muestra$gramaje_tela)
estadistico_shapiro_gramaje <- unname(prueba_shapiro_gramaje$statistic)
p_valor_shapiro_gramaje <- prueba_shapiro_gramaje$p.value
# END SOLUTION
prueba_shapiro_gramaje

In [ ]:
testthat::expect_true(exists("prueba_shapiro_gramaje"), info = "No encuentro la variable prueba_shapiro_gramaje.")

In [ ]:
testthat::expect_true(is.numeric(estadistico_shapiro_gramaje) && length(estadistico_shapiro_gramaje) == 1, info = "estadistico_shapiro_gramaje debe ser un solo número.")
testthat::expect_equal(as.numeric(estadistico_shapiro_gramaje), as.numeric(shapiro.test(muestra$gramaje_tela)$statistic), tolerance = 1e-6)

In [ ]:
testthat::expect_true(is.numeric(p_valor_shapiro_gramaje) && length(p_valor_shapiro_gramaje) == 1, info = "p_valor_shapiro_gramaje debe ser un solo número.")
testthat::expect_equal(as.numeric(p_valor_shapiro_gramaje), as.numeric(shapiro.test(muestra$gramaje_tela)$p.value), tolerance = 1e-6)

**Pregunta 1.2 (opción múltiple):** con un nivel de significancia del 5%, ¿qué puede asumir la empresa sobre la distribución del gramaje?

- A) El resultado depende del tamaño de muestra, no del p-valor obtenido.
- B) El p-valor alto no permite rechazar; es razonable asumir normalidad.
- C) La prueba de Shapiro-Wilk nunca permite concluir sobre normalidad.
- D) El p-valor bajo lleva a rechazar la normalidad de esta variable.

Guarda la letra en `respuesta_mcq1`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq1 <- if (shapiro.test(muestra$gramaje_tela)$p.value >= 0.05) "B" else "D"  # en la mayoría de las muestras: "B"
# END SOLUTION
respuesta_mcq1

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq1"), info = "Guarde su respuesta en respuesta_mcq1.")
testthat::expect_true(is.character(respuesta_mcq1) && toupper(trimws(respuesta_mcq1)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq1)), if (shapiro.test(muestra$gramaje_tela)$p.value >= 0.05) "B" else "D")

**Pregunta 1.3 (opción múltiple):** ¿qué implica esta conclusión para la prueba de medias que se hará a continuación?

- A) No es posible aplicar ninguna prueba de medias a esta variable.
- B) Se debe aplicar directamente una prueba no paramétrica en su lugar.
- C) Es válido usar una prueba t, pues el supuesto de normalidad se sostiene.
- D) El resultado de la prueba de medias ya no depende de los datos.

Guarda la letra en `respuesta_mcq2`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq2 <- if (shapiro.test(muestra$gramaje_tela)$p.value >= 0.05) "C" else "B"  # en la mayoría de las muestras: "C"
# END SOLUTION
respuesta_mcq2

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq2"), info = "Guarde su respuesta en respuesta_mcq2.")
testthat::expect_true(is.character(respuesta_mcq2) && toupper(trimws(respuesta_mcq2)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq2)), if (shapiro.test(muestra$gramaje_tela)$p.value >= 0.05) "C" else "B")

**Pregunta 1.4 (código):** con el supuesto de normalidad evaluado, contrasta si el gramaje promedio de tu muestra es consistente con el valor de 180 g/m² exigido en el contrato
$$H_0: \mu = 180,\quad H_1: \mu ≠ 180.$$
Aplica una prueba t de una muestra y guarda el resultado completo en `prueba_t_gramaje`, el estadístico en `estadistico_t_gramaje` y el p-valor en `p_valor_t_gramaje`.

**Función recomendada:** `t.test()` — [documentación](https://www.rdocumentation.org/packages/stats/versions/3.6.2/topics/t.test).


In [ ]:
# BEGIN SOLUTION
prueba_t_gramaje <- t.test(muestra$gramaje_tela, mu = 180)
estadistico_t_gramaje <- unname(prueba_t_gramaje$statistic)
p_valor_t_gramaje <- prueba_t_gramaje$p.value
# END SOLUTION
prueba_t_gramaje

In [ ]:
testthat::expect_true(exists("prueba_t_gramaje"), info = "No encuentro la variable prueba_t_gramaje.")

In [ ]:
testthat::expect_true(is.numeric(estadistico_t_gramaje) && length(estadistico_t_gramaje) == 1, info = "estadistico_t_gramaje debe ser un solo número.")
testthat::expect_equal(as.numeric(estadistico_t_gramaje), as.numeric(t.test(muestra$gramaje_tela, mu = 180)$statistic), tolerance = 1e-6)

In [ ]:
testthat::expect_true(is.numeric(p_valor_t_gramaje) && length(p_valor_t_gramaje) == 1, info = "p_valor_t_gramaje debe ser un solo número.")
testthat::expect_equal(as.numeric(p_valor_t_gramaje), as.numeric(t.test(muestra$gramaje_tela, mu = 180)$p.value), tolerance = 1e-6)

**Pregunta 1.5 (código):** la prueba de hipótesis solo indica si hay o no una diferencia significativa, pero no qué tan grande podría ser esa diferencia, para eso sirve el intervalo de confianza. A partir del objeto `prueba_t_gramaje` que ya calculaste, extrae el intervalo de confianza del 95% para la media poblacional del gramaje: guarda el límite inferior en `ic_inferior_gramaje` y el superior en `ic_superior_gramaje`.

In [ ]:
# BEGIN SOLUTION
ic_inferior_gramaje <- prueba_t_gramaje$conf.int[1]
ic_superior_gramaje <- prueba_t_gramaje$conf.int[2]
# END SOLUTION
c(ic_inferior_gramaje, ic_superior_gramaje)

In [ ]:
testthat::expect_true(is.numeric(ic_inferior_gramaje) && length(ic_inferior_gramaje) == 1, info = "ic_inferior_gramaje debe ser un solo número.")
testthat::expect_equal(as.numeric(ic_inferior_gramaje), as.numeric(t.test(muestra$gramaje_tela, mu = 180)$conf.int[1]), tolerance = 1e-6)
testthat::expect_true(is.numeric(ic_superior_gramaje) && length(ic_superior_gramaje) == 1, info = "ic_superior_gramaje debe ser un solo número.")
testthat::expect_equal(as.numeric(ic_superior_gramaje), as.numeric(t.test(muestra$gramaje_tela, mu = 180)$conf.int[2]), tolerance = 1e-6)

**Pregunta 1.6 (opción múltiple):** con base en la prueba t y el intervalo de confianza, ¿qué debería hacer la empresa?

- A) No actuar: el intervalo de confianza incluye el valor 180 exigido.
- B) Detener toda la planta y cambiar la maquinaria sin evaluar el costo.
- C) Ignorar el resultado, pues el p-valor no sirve para decidir nada.
- D) Reconocer la diferencia y evaluar si justifica recalibrar la máquina.

Guarda la letra en `respuesta_mcq3`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq3 <- if (t.test(muestra$gramaje_tela, mu = 180)$p.value < 0.05) "D" else "A"  # en la mayoría de las muestras: "D"
# END SOLUTION
respuesta_mcq3

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq3"), info = "Guarde su respuesta en respuesta_mcq3.")
testthat::expect_true(is.character(respuesta_mcq3) && toupper(trimws(respuesta_mcq3)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq3)), if (t.test(muestra$gramaje_tela, mu = 180)$p.value < 0.05) "D" else "A")

**Pregunta 1.7 (opción múltiple):** ¿cuál afirmación describe mejor la diferencia entre significancia estadística y relevancia práctica en este caso?

- A) Son equivalentes: toda diferencia significativa es relevante.
- B) Una diferencia puede ser significativa sin ser relevante en la práctica.
- C) La relevancia práctica se mide únicamente con el valor del p-valor.
- D) Si no es significativa, entonces sí es relevante para el negocio.

Guarda la letra en `respuesta_mcq4`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq4 <- "B"
# END SOLUTION

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq4"), info = "Guarde su respuesta en respuesta_mcq4.")
testthat::expect_true(is.character(respuesta_mcq4) && toupper(trimws(respuesta_mcq4)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq4)), "B")

---
## Parte 2 — ¿Cuántos defectos de costura son "normales"?

El equipo de calidad cuenta el número de defectos visibles de costura en cada prenda. El contrato establece que, en una prenda típica, no debería haber más de un defecto. Al ser un conteo, conviene verificar primero si es razonable tratarla como aproximadamente normal, igual que en la Parte 1.

**Función recomendada:** `shapiro.test()` (la misma de la Parte 1).


**Pregunta 2.1 (código):** antes de elegir una prueba para contrastar la mediana de defectos, verifica si es razonable asumir que `defectos_costura` se distribuye normalmente (aunque, al ser un conteo, es probable que no lo sea). Aplica Shapiro-Wilk y guarda el resultado completo en `prueba_shapiro_defectos`, el estadístico en `estadistico_shapiro_defectos` y el p-valor en `p_valor_shapiro_defectos`.

In [ ]:
# BEGIN SOLUTION
prueba_shapiro_defectos <- shapiro.test(muestra$defectos_costura)
estadistico_shapiro_defectos <- unname(prueba_shapiro_defectos$statistic)
p_valor_shapiro_defectos <- prueba_shapiro_defectos$p.value
# END SOLUTION
prueba_shapiro_defectos

In [ ]:
testthat::expect_true(exists("prueba_shapiro_defectos"), info = "No encuentro la variable prueba_shapiro_defectos.")

In [ ]:
testthat::expect_true(is.numeric(estadistico_shapiro_defectos) && length(estadistico_shapiro_defectos) == 1, info = "estadistico_shapiro_defectos debe ser un solo número.")
testthat::expect_equal(as.numeric(estadistico_shapiro_defectos), as.numeric(shapiro.test(muestra$defectos_costura)$statistic), tolerance = 1e-6)

In [ ]:
testthat::expect_true(is.numeric(p_valor_shapiro_defectos) && length(p_valor_shapiro_defectos) == 1, info = "p_valor_shapiro_defectos debe ser un solo número.")
testthat::expect_equal(as.numeric(p_valor_shapiro_defectos), as.numeric(shapiro.test(muestra$defectos_costura)$p.value), tolerance = 1e-8)

Con un conteo de defectos (valores enteros, no negativos, frecuentemente asimétricos), es común que la distribución no sea normal (tu p-valor debería confirmarlo). Dado que no se puede asumir normalidad, la alternativa es una prueba no paramétrica que no dependa de ese supuesto.

**Función recomendada:** `wilcox.test()` — [documentación](https://www.rdocumentation.org/packages/stats/versions/3.6.2/topics/wilcox.test).


**Pregunta 2.2 (código):** aplica una prueba de Wilcoxon de una muestra para contrastar si la mediana de `defectos_costura` ($\eta$) es a lo sumo 1, el máximo aceptable según el contrato

$$H0: \eta \leq 1, \quad H_1: \eta > 1 $$

Guarda el resultado completo en `prueba_wilcox_defectos`, el estadístico en `estadistico_wilcox_defectos` y el p-valor en `p_valor_wilcox_defectos`.

In [ ]:
# BEGIN SOLUTION
prueba_wilcox_defectos <- wilcox.test(muestra$defectos_costura, mu = 1, alternative = "greater")
estadistico_wilcox_defectos <- unname(prueba_wilcox_defectos$statistic)
p_valor_wilcox_defectos <- prueba_wilcox_defectos$p.value
# END SOLUTION
prueba_wilcox_defectos

In [ ]:
testthat::expect_true(exists("prueba_wilcox_defectos"), info = "No encuentro la variable prueba_wilcox_defectos.")

In [ ]:
ref <- suppressWarnings(wilcox.test(muestra$defectos_costura, mu = 1, alternative = "greater"))
testthat::expect_true(is.numeric(estadistico_wilcox_defectos) && length(estadistico_wilcox_defectos) == 1, info = "estadistico_wilcox_defectos debe ser un solo número.")
testthat::expect_equal(as.numeric(estadistico_wilcox_defectos), as.numeric(ref$statistic), tolerance = 1e-6)

In [ ]:
ref <- suppressWarnings(wilcox.test(muestra$defectos_costura, mu = 1, alternative = "greater"))
testthat::expect_true(is.numeric(p_valor_wilcox_defectos) && length(p_valor_wilcox_defectos) == 1, info = "p_valor_wilcox_defectos debe ser un solo número.")
testthat::expect_equal(as.numeric(p_valor_wilcox_defectos), as.numeric(ref$p.value), tolerance = 1e-8)

**Pregunta 2.3 (opción múltiple):** ¿qué debería concluir y hacer la empresa?

- A) No se requiere ninguna acción: el proceso cumple el estándar pactado.
- B) La prueba de Wilcoxon no permite concluir nada sobre la mediana real.
- C) El resultado indica que se debe modificar el estándar del contrato.
- D) Hay evidencia de que se supera el estándar; conviene reforzar la inspección.

Guarda la letra en `respuesta_mcq5`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq5 <- if (suppressWarnings(wilcox.test(muestra$defectos_costura, mu = 1, alternative = "greater"))$p.value < 0.05) "D" else "A"  # en la mayoría de las muestras: "D"
# END SOLUTION
respuesta_mcq5

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq5"), info = "Guarde su respuesta en respuesta_mcq5.")
testthat::expect_true(is.character(respuesta_mcq5) && toupper(trimws(respuesta_mcq5)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq5)), if (suppressWarnings(wilcox.test(muestra$defectos_costura, mu = 1, alternative = "greater"))$p.value < 0.05) "D" else "A")

**Pregunta 2.4 (opción múltiple):** ¿cuál es una limitación real de la prueba de Wilcoxon en este caso?

- A) Los muchos empates en datos de conteo afectan la precisión del p-valor.
- B) Requiere que los datos sean normales, igual que la prueba t.
- C) Nunca puede aplicarse a variables que sean de tipo conteo.
- D) No presenta ninguna limitación frente a la prueba t de medias.

Guarda la letra en `respuesta_mcq6`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq6 <- "A"
# END SOLUTION

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq6"), info = "Guarde su respuesta en respuesta_mcq6.")
testthat::expect_true(is.character(respuesta_mcq6) && toupper(trimws(respuesta_mcq6)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq6)), "A")

---
## Parte 3 — ¿Se cumple la conformidad de color pactada en el contrato?

El contrato exige que al menos el 95% de las prendas tengan un color dentro de la tolerancia aprobada. Si la conformidad real cae por debajo, la empresa se expone a una multa contractual o a que el cliente rechace el lote.

**Función recomendada:** `prop.test()` — [documentación](https://www.rdocumentation.org/packages/stats/versions/3.6.2/topics/prop.test).


**Pregunta 3.1 (código):** el primer paso es calcular cuántas prendas de tu muestra cumplen la tolerancia de color (`color_conforme` sea `"Si"`). Calcula este valor en la variable `n_conforme`, junto con el número total de muestras `n_total`. 

Después, a partir de ese conteo, contrastar si la proporción real es consistente con el 95% exigido en el contrato 
$$H_0: \pi \geq 0.95, \quad H_1: \pi < 0.95 $$

Guarda el resultado completo de `prop.test()` en `prueba_prop_color`, el estadístico en `estadistico_prop_color` y el p-valor en `p_valor_prop_color`.

In [ ]:
# BEGIN SOLUTION
n_conforme <- sum(muestra$color_conforme == "Si")
n_total <- nrow(muestra)
prueba_prop_color <- prop.test(n_conforme, n_total, p = 0.95, alternative = "less")
estadistico_prop_color <- unname(prueba_prop_color$statistic)
p_valor_prop_color <- prueba_prop_color$p.value
# END SOLUTION
prueba_prop_color

In [ ]:
testthat::expect_true(is.numeric(n_conforme) && length(n_conforme) == 1, info = "n_conforme debe ser un solo número.")
testthat::expect_equal(as.numeric(n_conforme), as.numeric(sum(muestra$color_conforme == "Si")), tolerance = 1e-6)
testthat::expect_true(is.numeric(n_total) && length(n_total) == 1, info = "n_total debe ser un solo número.")
testthat::expect_equal(as.numeric(n_total), as.numeric(nrow(muestra)), tolerance = 1e-6)

In [ ]:
r1 <- prop.test(sum(muestra$color_conforme == "Si"), nrow(muestra), p = 0.95, alternative = "less")
r2 <- prop.test(sum(muestra$color_conforme == "Si"), nrow(muestra), p = 0.95, alternative = "less", correct = FALSE)
testthat::expect_true(isTRUE(all.equal(p_valor_prop_color, r1$p.value, tolerance = 1e-6)) || isTRUE(all.equal(p_valor_prop_color, r2$p.value, tolerance = 1e-6)), info = "Revise los argumentos p y alternative de prop.test().")
testthat::expect_true(isTRUE(all.equal(as.numeric(estadistico_prop_color), unname(r1$statistic), tolerance = 1e-6)) || isTRUE(all.equal(as.numeric(estadistico_prop_color), unname(r2$statistic), tolerance = 1e-6)))

**Pregunta 3.2 (código):** al igual que con el gramaje, la prueba de proporción solo indica si hay o no diferencia con el estándar, pero el intervalo de confianza permite dimensionarla, y en este caso, comparar dos formas distintas de construirlo. Calcula el intervalo de confianza del 95% para la proporción de conformidad de color usando:

- **Wald** (fórmula manual): `p̂ ± 1.96 * sqrt(p̂(1-p̂)/n)`.
- **Wilson**: con `prop.test(n_conforme, n_total, correct = FALSE)`.

Guarda: `ic_wald_inferior`, `ic_wald_superior`, `ic_wilson_inferior`, `ic_wilson_superior`.


In [ ]:
# BEGIN SOLUTION
p_hat <- n_conforme / n_total
error_wald <- 1.96 * sqrt(p_hat * (1 - p_hat) / n_total)
ic_wald_inferior <- p_hat - error_wald
ic_wald_superior <- p_hat + error_wald
ic_wilson <- prop.test(n_conforme, n_total, correct = FALSE)$conf.int
ic_wilson_inferior <- ic_wilson[1]
ic_wilson_superior <- ic_wilson[2]
# END SOLUTION
c(ic_wald_inferior, ic_wald_superior, ic_wilson_inferior, ic_wilson_superior)

In [ ]:
p <- sum(muestra$color_conforme == "Si") / nrow(muestra); e <- 1.96 * sqrt(p * (1 - p) / nrow(muestra))
testthat::expect_true(is.numeric(ic_wald_inferior) && length(ic_wald_inferior) == 1, info = "ic_wald_inferior debe ser un solo número.")
testthat::expect_equal(as.numeric(ic_wald_inferior), as.numeric(p - e), tolerance = 1e-6)
testthat::expect_true(is.numeric(ic_wald_superior) && length(ic_wald_superior) == 1, info = "ic_wald_superior debe ser un solo número.")
testthat::expect_equal(as.numeric(ic_wald_superior), as.numeric(p + e), tolerance = 1e-6)

In [ ]:
w <- prop.test(sum(muestra$color_conforme == "Si"), nrow(muestra), correct = FALSE)$conf.int
testthat::expect_true(is.numeric(ic_wilson_inferior) && length(ic_wilson_inferior) == 1, info = "ic_wilson_inferior debe ser un solo número.")
testthat::expect_equal(as.numeric(ic_wilson_inferior), as.numeric(w[1]), tolerance = 1e-6)
testthat::expect_true(is.numeric(ic_wilson_superior) && length(ic_wilson_superior) == 1, info = "ic_wilson_superior debe ser un solo número.")
testthat::expect_equal(as.numeric(ic_wilson_superior), as.numeric(w[2]), tolerance = 1e-6)

**Pregunta 3.3 (opción múltiple):** ¿cuál es la interpretación correcta del intervalo de confianza del 95%?

- A) Hay 95% de probabilidad de que el valor real esté en este intervalo.
- B) El intervalo garantiza que el 95% de las prendas futuras cumplirán.
- C) El intervalo construido no tiene ninguna interpretación probabilística.
- D) En el 95% de muestras repetidas, el intervalo contendría el valor real.

Guarda la letra en `respuesta_mcq7`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq7 <- "D"
# END SOLUTION

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq7"), info = "Guarde su respuesta en respuesta_mcq7.")
testthat::expect_true(is.character(respuesta_mcq7) && toupper(trimws(respuesta_mcq7)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq7)), "D")

**Pregunta 3.4 (opción múltiple):** ¿qué debería concluir y hacer la empresa?

- A) 0.95 queda por encima del intervalo de Wilson: hay evidencia de incumplimiento.
- B) 0.95 queda dentro del intervalo de Wilson: no hay evidencia suficiente de incumplimiento.
- C) Los dos métodos son contradictorios, así que no se puede decidir.
- D) El intervalo de confianza no aporta nada para esta decisión.

Guarda la letra en `respuesta_mcq8`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq8 <- if (prop.test(sum(muestra$color_conforme == "Si"), nrow(muestra), correct = FALSE)$conf.int[2] < 0.95) "A" else "B"  # en la mayoría de las muestras: "A"
# END SOLUTION
respuesta_mcq8

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq8"), info = "Guarde su respuesta en respuesta_mcq8.")
testthat::expect_true(is.character(respuesta_mcq8) && toupper(trimws(respuesta_mcq8)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq8)), if (prop.test(sum(muestra$color_conforme == "Si"), nrow(muestra), correct = FALSE)$conf.int[2] < 0.95) "A" else "B")

**Pregunta 3.5 (opción múltiple):** ¿cuál describe mejor la diferencia entre el método de Wald y el de Wilson?

- A) Los dos métodos siempre producen exactamente el mismo intervalo.
- B) Wald puede fallar en casos extremos; Wilson corrige ese problema.
- C) Wilson no puede calcularse con las funciones disponibles en R.
- D) Wald es siempre más preciso que Wilson en cualquier situación.

Guarda la letra en `respuesta_mcq9`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq9 <- "B"
# END SOLUTION

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq9"), info = "Guarde su respuesta en respuesta_mcq9.")
testthat::expect_true(is.character(respuesta_mcq9) && toupper(trimws(respuesta_mcq9)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq9)), "B")

**Pregunta 3.6 (código):** el error estándar mide qué tan precisa es tu estimación de la proporción, y es el mismo valor que usaste como base del método de Wald. Calcúlalo y guárdalo en `error_estandar_color`.

In [ ]:
# BEGIN SOLUTION
error_estandar_color <- sqrt(p_hat * (1 - p_hat) / n_total)
# END SOLUTION
error_estandar_color

In [ ]:
testthat::expect_true(is.numeric(error_estandar_color) && length(error_estandar_color) == 1, info = "error_estandar_color debe ser un solo número.")
testthat::expect_equal(as.numeric(error_estandar_color), as.numeric(sqrt((sum(muestra$color_conforme == "Si")/nrow(muestra)) * (1 - sum(muestra$color_conforme == "Si")/nrow(muestra)) / nrow(muestra))), tolerance = 1e-6)

**Pregunta 3.7 (opción múltiple):** si el gerente de calidad quería estimar la conformidad de color con un **error máximo de 0.025 (2.5 puntos porcentuales)** al 95% de confianza, y comparas eso con el margen de error realmente logrado (`1.96 × error_estandar_color`), ¿el estudio quedó bien diseñado para ese objetivo? *(No te bases en qué tan grande te parezca el tamaño de muestra — compara los números.)*

- A) Sí, porque 150 prendas es una muestra grande que garantiza precisión.
- B) Sí, porque el margen logrado es menor o igual al 0.025 deseado.
- C) No, porque el margen logrado supera el 0.025 que se quería alcanzar.
- D) No se puede saber sin conocer la proporción poblacional real.

Guarda la letra en `respuesta_mcq10`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq10 <- {p <- sum(muestra$color_conforme == "Si")/nrow(muestra); if (1.96 * sqrt(p*(1-p)/nrow(muestra)) > 0.025) "C" else "B"}  # en la mayoría de las muestras: "C"
# END SOLUTION
respuesta_mcq10

In [ ]:
. = " # BEGIN TEST CONFIG
points: 0
" # END TEST CONFIG
testthat::expect_true(exists("respuesta_mcq10"), info = "Guarde su respuesta en respuesta_mcq10.")
testthat::expect_true(is.character(respuesta_mcq10) && toupper(trimws(respuesta_mcq10)) %in% c("A", "B", "C", "D"), info = "La respuesta debe ser una letra entre A y D, por ejemplo \"B\".")

In [ ]:
. = " # BEGIN TEST CONFIG
hidden: true
points: 1
" # END TEST CONFIG
testthat::expect_equal(toupper(trimws(respuesta_mcq10)), {p <- sum(muestra$color_conforme == "Si")/nrow(muestra); if (1.96 * sqrt(p*(1-p)/nrow(muestra)) > 0.025) "C" else "B"})

---
## Cierre — diagnóstico y recomendación

**Pregunta final (reflexión, calificada manualmente):** con base en los tres análisis anteriores, escribe **un párrafo** identificando los problemas de calidad que enfrenta la empresa y qué acciones concretas recomendarías tomar, considerando las alternativas y sus costos (recalibrar la máquina tejedora vs. ajustar la tensión del telar; contratar un inspector adicional vs. mantenimiento preventivo; asumir la multa contractual vs. negociar una tolerancia de color con descuento).

**Respuesta modelo:** Un buen párrafo identifica los **tres problemas**: (1) el gramaje medio está por debajo de 180 g/m² (la diferencia es significativa, aunque su magnitud —pocos g/m²— debe compararse con la tolerancia técnica antes de decidir); (2) la mediana de defectos de costura supera el máximo de 1 por prenda; (3) la conformidad de color está claramente por debajo del 95 % (el intervalo completo queda bajo 0,95). Luego **prioriza con criterio de costo**: el color es el incumplimiento más grave y más costoso (multa o rechazo de lotes), por lo que conviene negociar con el cliente una tolerancia con descuento mientras se corrige el proceso de tinturado; para el gramaje, ajustar la tensión del telar suele ser más barato que recalibrar la máquina y debe intentarse primero; para los defectos de costura, el mantenimiento preventivo ataca la causa, mientras que un inspector adicional solo detecta el problema. Debe mencionar que la muestra de 150 prendas no alcanza la precisión de ±0,025 deseada para la proporción.

---
## Entrega

1. Verifica que corriste todas las celdas en orden y que no hay errores.
2. Descarga el notebook: `Archivo > Descargar > Descargar .ipynb`.
3. Sube el archivo a Moodle con el nombre `E2-02_ControlCalidad_<tu_codigo>.ipynb`.